<a href="https://colab.research.google.com/github/tothanhha2006-beep/Tuan3a/blob/main/Tuan3a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Cau1**

In [18]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, f1_score, average_precision_score

# Tải dữ liệu gốc
cancer = load_breast_cancer(as_frame=True)
X_all = cancer.data
y_all = cancer.target  # 0: Malignant, 1: Benign

df_all = X_all.copy()
df_all["target"] = y_all

# Phân tách lành tính (Benign - 1) và ác tính (Malignant - 0)
benign_df = df_all[df_all["target"] == 1]
malignant_all = df_all[df_all["target"] == 0]

RANDOM_STATE = 42
scenarios = {
    '40 Malignant': 40,
    '120 Malignant': 120,
    'Full Dataset': len(malignant_all)
}

comparison_results = []

for label, count in scenarios.items():
    if label == 'Full Dataset':
        malignant_sub = malignant_all
    else:
        malignant_sub = malignant_all.sample(n=count, random_state=RANDOM_STATE)

    # Kết hợp các mẫu malignant với toàn bộ mẫu benign
    df_scenario = pd.concat([benign_df, malignant_sub], axis=0)

    X = df_scenario.drop(columns=["target"])
    y = df_scenario["target"]

    # Chia tập Train-Test với tỷ lệ 80:20 (stratify theo nhãn)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    # Chuẩn hóa dữ liệu
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Huấn luyện Logistic Regression
    lr_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
    lr_model.fit(X_train_scaled, y_train)

    # Dự đoán lớp và xác suất
    y_pred = lr_model.predict(X_test_scaled)
    # Lưu ý: Lớp dương tính cần đánh giá ở đây là Malignant (nhãn 0)
    y_proba = lr_model.predict_proba(X_test_scaled)[:, 0]

    # Tính toán các chỉ số với lớp dương tính (pos_label) là Malignant (0)
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, pos_label=0)
    f1 = f1_score(y_test, y_pred, pos_label=0)
    # pr_auc cho nhãn 0
    pr_auc = average_precision_score(1 - y_test, y_proba)

    comparison_results.append({
        "Scenario": label,
        "Malignant Count": count,
        "Benign Count": len(benign_df),
        "Accuracy": f"{accuracy:.4f}",
        "Recall": f"{recall:.4f}",
        "F1-Score": f"{f1:.4f}",
        "PR-AUC": f"{pr_auc:.4f}"
    })

df_comparison = pd.DataFrame(comparison_results)
display(df_comparison)

,Scenario,Malignant Count,Benign Count,Accuracy,Recall,F1-Score,PR-AUC
0,40 Malignant,40,357,0.9875,0.8750,0.9333,0.9375
1,120 Malignant,120,357,0.9688,0.8750,0.9333,0.9679
2,Full Dataset,212,357,0.9825,0.9762,0.9762,0.9940


# **Cau2**

In [22]:
kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

stratified_kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [23]:
def get_fold_distribution(cv, X_data, y_data, cv_name):
    rows = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_data, y_data), start=1):
        y_fold_train = y_data.iloc[train_idx]
        y_fold_val = y_data.iloc[val_idx]

        rows.append({
            "CV": cv_name,
            "Fold": fold,
            "Train malignant (%)": 100 * y_fold_train.mean(),
            "Validation malignant (%)": 100 * y_fold_val.mean(),
            "Validation size": len(val_idx)
        })

    return pd.DataFrame(rows)

dist_kfold = get_fold_distribution(
    kfold, X_train.reset_index(drop=True), y_train.reset_index(drop=True), "KFold"
)

dist_skfold = get_fold_distribution(
    stratified_kfold,
    X_train.reset_index(drop=True),
    y_train.reset_index(drop=True),
    "StratifiedKFold"
)

fold_distribution = pd.concat(
    [dist_kfold, dist_skfold],
    ignore_index=True
)

display(fold_distribution.round(2))

,CV,Fold,Train malignant (%),Validation malignant (%),Validation size
0,KFold,1,63.19,60.44,91
1,KFold,2,60.99,69.23,91
2,KFold,3,65.11,52.75,91
3,KFold,4,60.71,70.33,91
4,KFold,5,63.19,60.44,91
5,StratifiedKFold,1,62.64,62.64,91
6,StratifiedKFold,2,62.64,62.64,91
7,StratifiedKFold,3,62.64,62.64,91
8,StratifiedKFold,4,62.64,62.64,91
9,StratifiedKFold,5,62.64,62.64,91


In [28]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, accuracy_score, recall_score, f1_score, precision_score
import pandas as pd

# Định nghĩa các scorer đánh giá tập trung vào lớp Malignant (nhãn 0)
scoring = {
    "accuracy": make_scorer(accuracy_score),
    "recall_malignant": make_scorer(recall_score, pos_label=0),
    "f1_malignant": make_scorer(f1_score, pos_label=0),
    "precision_malignant": make_scorer(precision_score, pos_label=0)
}

baseline_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])

baseline_cv = cross_validate(
    baseline_pipeline,
    X_train,
    y_train,
    cv=stratified_kfold,
    scoring=scoring
)

baseline_cv_table = pd.DataFrame({
    metric: baseline_cv[f"test_{metric}"]
    for metric in scoring.keys()
})

baseline_cv_table.index = [
    f"Fold {i}" for i in range(1, len(baseline_cv_table) + 1)
]

display(baseline_cv_table.round(4))

,accuracy,recall_malignant,f1_malignant,precision_malignant
Fold 1,0.967,0.9118,0.9538,1.0000
Fold 2,0.989,1.0000,0.9855,0.9714
Fold 3,0.978,0.9706,0.9706,0.9706
Fold 4,0.989,0.9706,0.9851,1.0000
Fold 5,0.967,0.9706,0.9565,0.9429


In [29]:
baseline_cv_result = pd.DataFrame({
    "Mean": baseline_cv_table.mean(),
    "Std": baseline_cv_table.std()
})

display(baseline_cv_result.round(4))

,Mean,Std
accuracy,0.9780,0.0110
recall_malignant,0.9647,0.0322
f1_malignant,0.9703,0.0151
precision_malignant,0.9770,0.0240


In [30]:
print("""
### 💡 TẠI SAO STRATIFIED KFOLD ỔN ĐỊNH HƠN KFOLD VỚI DỮ LIỆU IMBALANCE?

Dựa trên kết quả phân phối thực nghiệm từ các fold ở trên, chúng ta có hai lý do chính cốt lõi giải thích cho hiện tượng này:

#### 1. Đảm bảo tỷ lệ nhãn đồng đều giữa các Folds (Class Proportion Preservation)
- **KFold truyền thống:** Chỉ thực hiện chia dữ liệu ngẫu nhiên (Blind Split) mà không màng tới nhãn lớp. Trên tập dữ liệu mất cân bằng nặng, một fold kiểm thử có thể ngẫu nhiên chứa cực kỳ ít mẫu thuộc lớp thiểu số (Malignant), hoặc thậm chí có thể **không chứa mẫu nào**.
- **StratifiedKFold:** Bảo toàn tỷ lệ nhãn ban đầu (ví dụ: ~62.6% lớp Malignant) đồng nhất trên tất cả 5 folds cho cả tập Train và tập Validation. Điều này giúp mô hình luôn đối mặt với một cấu trúc bài toán nhất quán giữa các lượt thử nghiệm.

#### 2. Giảm thiểu biến động điểm số (Low Variance & Standard Deviation)
- Với **KFold**, vì phân phối dữ liệu bị lệch ngẫu nhiên qua từng lượt huấn luyện, mô hình kiểm định sẽ cho điểm số biến động rất mạnh (độ lệch chuẩn cao) do tập validation ở một số fold quá dễ hoặc quá thiếu mẫu thiểu số để đánh giá chính xác.
- Với **StratifiedKFold**, nhờ tính nhất quán của tập validation, điểm đánh giá qua các fold đại diện thực tế tốt hơn năng lực tổng quát hóa của mô hình, dẫn tới **Mean Score phản ánh chính xác** thực tế và **Độ lệch chuẩn (Standard Deviation) nhỏ hơn đáng kể** (mô hình ổn định hơn).
""")


### 💡 TẠI SAO STRATIFIED KFOLD ỔN ĐỊNH HƠN KFOLD VỚI DỮ LIỆU IMBALANCE?

Dựa trên kết quả phân phối thực nghiệm từ các fold ở trên, chúng ta có hai lý do chính cốt lõi giải thích cho hiện tượng này:

#### 1. Đảm bảo tỷ lệ nhãn đồng đều giữa các Folds (Class Proportion Preservation)
- **KFold truyền thống:** Chỉ thực hiện chia dữ liệu ngẫu nhiên (Blind Split) mà không màng tới nhãn lớp. Trên tập dữ liệu mất cân bằng nặng, một fold kiểm thử có thể ngẫu nhiên chứa cực kỳ ít mẫu thuộc lớp thiểu số (Malignant), hoặc thậm chí có thể **không chứa mẫu nào**.
- **StratifiedKFold:** Bảo toàn tỷ lệ nhãn ban đầu (ví dụ: ~62.6% lớp Malignant) đồng nhất trên tất cả 5 folds cho cả tập Train và tập Validation. Điều này giúp mô hình luôn đối mặt với một cấu trúc bài toán nhất quán giữa các lượt thử nghiệm.

#### 2. Giảm thiểu biến động điểm số (Low Variance & Standard Deviation)
- Với **KFold**, vì phân phối dữ liệu bị lệch ngẫu nhiên qua từng lượt huấn luyện, mô hình kiểm định sẽ cho điểm số biến độn

# **Cau3**

In [31]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate

k_values = [3, 5, 10]
summary_rows = []

for k in k_values:
    # Thiết lập StratifiedKFold với K tương ứng
    cv_stratified = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)

    # Tiến hành kiểm định chéo
    cv_results = cross_validate(
        baseline_pipeline,
        X_train,
        y_train,
        cv=cv_stratified,
        scoring=scoring
    )

    # Tính toán Mean và Std cho từng chỉ số
    row_mean = {f"{metric}_mean": np.mean(cv_results[f"test_{metric}"]) for metric in scoring.keys()}
    row_std = {f"{metric}_std": np.std(cv_results[f"test_{metric}"]) for metric in scoring.keys()}

    combined_row = {"K": k}
    combined_row.update(row_mean)
    combined_row.update(row_std)
    summary_rows.append(combined_row)

# Tạo bảng tổng hợp kết quả
df_k_comparison = pd.DataFrame(summary_rows)

# Định dạng lại bảng hiển thị trực quan hơn
formatted_rows = []
for r in summary_rows:
    formatted_rows.append({
        "Số Fold (K)": r["K"],
        "Accuracy (Mean ± Std)": f"{r['accuracy_mean']:.4f} ± {r['accuracy_std']:.4f}",
        "F1 Malignant (Mean ± Std)": f"{r['f1_malignant_mean']:.4f} ± {r['f1_malignant_std']:.4f}",
        "Recall Malignant (Mean ± Std)": f"{r['recall_malignant_mean']:.4f} ± {r['recall_malignant_std']:.4f}"
    })

df_formatted = pd.DataFrame(formatted_rows)
display(df_formatted)

print("""
### 🔎 QUAN SÁT SỰ THAY ĐỔI KHI THAY ĐỔI SỐ FOLDS (K):

1. **Ảnh hưởng tới Mean (Giá trị trung bình):**
   - Điểm trung bình (Mean) của các chỉ số (Accuracy, F1-Score) có xu hướng duy trì khá ổn định và có thể tăng nhẹ khi K tăng từ 3 lên 10.
   - Lý do: Khi K tăng, tập huấn luyện (Train set) ở mỗi fold sẽ lớn hơn (chiếm tỉ lệ (K-1)/K của tập dữ liệu ban đầu). Do đó, mô hình học được nhiều thông tin hơn ở mỗi fold, giúp điểm số trung bình cải thiện.

2. **Ảnh hưởng tới Standard Deviation (Độ lệch chuẩn - Std):**
   - Điểm số dao động (Std) thường sẽ có xu hướng thay đổi. Khi K nhỏ (K = 3), mỗi fold validation có kích thước lớn hơn, điểm số giữa các fold ít bị nhiễu cục bộ. Khi K lớn (K = 10), tập validation ở mỗi fold trở nên rất nhỏ (chỉ chứa khoảng 10% dữ liệu), khiến kết quả đánh giá trên từng fold dễ bị ảnh hưởng bởi vài mẫu cá biệt, đôi khi dẫn đến tăng độ biến động (Std) giữa các fold.
""")

,Số Fold (K),Accuracy (Mean ± Std),F1 Malignant (Mean ± Std),Recall Malignant (Mean ± Std)
0,3,0.9758 ± 0.0030,0.9673 ± 0.0044,0.9590 ± 0.0218
1,5,0.9780 ± 0.0098,0.9703 ± 0.0135,0.9647 ± 0.0288
2,10,0.9824 ± 0.0133,0.9764 ± 0.0172,0.9706 ± 0.0294



### 🔎 QUAN SÁT SỰ THAY ĐỔI KHI THAY ĐỔI SỐ FOLDS (K):

1. **Ảnh hưởng tới Mean (Giá trị trung bình):**
   - Điểm trung bình (Mean) của các chỉ số (Accuracy, F1-Score) có xu hướng duy trì khá ổn định và có thể tăng nhẹ khi K tăng từ 3 lên 10.
   - Lý do: Khi K tăng, tập huấn luyện (Train set) ở mỗi fold sẽ lớn hơn (chiếm tỉ lệ (K-1)/K của tập dữ liệu ban đầu). Do đó, mô hình học được nhiều thông tin hơn ở mỗi fold, giúp điểm số trung bình cải thiện.

2. **Ảnh hưởng tới Standard Deviation (Độ lệch chuẩn - Std):**
   - Điểm số dao động (Std) thường sẽ có xu hướng thay đổi. Khi K nhỏ (K = 3), mỗi fold validation có kích thước lớn hơn, điểm số giữa các fold ít bị nhiễu cục bộ. Khi K lớn (K = 10), tập validation ở mỗi fold trở nên rất nhỏ (chỉ chứa khoảng 10% dữ liệu), khiến kết quả đánh giá trên từng fold dễ bị ảnh hưởng bởi vài mẫu cá biệt, đôi khi dẫn đến tăng độ biến động (Std) giữa các fold.



# **Cau4**

In [39]:
import pandas as pd
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# Khởi tạo Pipeline tích hợp cả việc chuẩn hóa, over-sampling và phân lớp
pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('sampler', RandomOverSampler(random_state=RANDOM_STATE)),
    ('classifier', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
])

scoring = ['recall', 'f1', 'average_precision']
k_values = [3, 5, 10]
results = []

# Vòng lặp khảo sát các giá trị K
for k in k_values:
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)

    # ĐÃ SỬA: Thay X_combined, y_combined bằng X_train, y_train hiện có trong Notebook
    scores = cross_validate(pipeline, X_train, y_train, cv=skf, scoring=scoring)

    # Tính tỷ lệ Train / Validation tương ứng
    train_pct = round((k - 1) / k * 100, 1)
    val_pct = round(1 / k * 100, 1)

    results.append({
        "Số lượng Fold": f"K = {k}",
        "Tỷ lệ Train/Val": f"{train_pct}% / {val_pct}%",
        "Recall (Mean ± Std)": f"{scores['test_recall'].mean():.4f} ± {scores['test_recall'].std():.4f}",
        "F1-score (Mean ± Std)": f"{scores['test_f1'].mean():.4f} ± {scores['test_f1'].std():.4f}",
        "PR-AUC (Mean ± Std)": f"{scores['test_average_precision'].mean():.4f} ± {scores['test_average_precision'].std():.4f}"
    })

df_k_results = pd.DataFrame(results)
print(df_k_results.to_string(index=False))



Số lượng Fold Tỷ lệ Train/Val Recall (Mean ± Std) F1-score (Mean ± Std) PR-AUC (Mean ± Std)
        K = 3   66.7% / 33.3%     0.9825 ± 0.0131       0.9824 ± 0.0067     0.9968 ± 0.0026
        K = 5   80.0% / 20.0%     0.9754 ± 0.0179       0.9771 ± 0.0120     0.9961 ± 0.0037
       K = 10   90.0% / 10.0%     0.9860 ± 0.0235       0.9842 ± 0.0125     0.9959 ± 0.0099


In [45]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# ĐÃ SỬA: Sử dụng chính xác các biến đã có ở Bài 1 (malignant_all, benign_df)
# Loại bỏ cột 'target' để giữ lại chỉ các đặc trưng (X)
X_benign_clean = benign_df.drop(columns=['target'])
X_malignant_clean = malignant_all.drop(columns=['target'])

n_pos = 60
X_sub_pos = X_malignant_clean.iloc[:n_pos]
X_combined = pd.concat([X_benign_clean, X_sub_pos], axis=0)
y_combined = np.array([1] * len(X_benign_clean) + [0] * len(X_sub_pos)) # 1: Benign (lớp đa số), 0: Malignant (lớp thiểu số)

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y_combined, test_size=0.2, stratify=y_combined, random_state=42
)
strategies = [0.5, 0.75, 1.0]
results = []

for strategy in strategies:
    # Chú ý: Ở đây tỷ lệ thiểu số (0) / đa số (1) là 60 / 357 ≈ 0.168
    # Do đó các strategy [0.5, 0.75, 1.0] đều lớn hơn 0.168 và hoạt động tốt!
    try:
        pipeline = ImbPipeline([
            ('scaler', StandardScaler()),
            ('sampler', RandomOverSampler(sampling_strategy=strategy, random_state=42)),
            ('classifier', LogisticRegression(random_state=42, max_iter=1000))
        ])

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]

        # Lấy số lượng mẫu sau khi oversampling trên tập train
        resampled_X, resampled_y = pipeline.named_steps['sampler'].fit_resample(
            pipeline.named_steps['scaler'].fit_transform(X_train), y_train
        )
        n_class_1 = sum(resampled_y == 1)
        n_class_0 = sum(resampled_y == 0)

        if strategy == 0.5:
            ratio_str = f"{n_class_1} : {n_class_0} (2 : 1)"
        elif strategy == 0.75:
            ratio_str = f"{n_class_1} : {n_class_0} (4 : 3)"
        else:
            ratio_str = f"{n_class_1} : {n_class_0} (1 : 1)"

        results.append({
            "sampling_strategy": strategy,
            "Tỷ lệ lớp 1 : lớp 0 sau sampling": ratio_str,
            "Precision (Nhãn 0)": round(precision_score(y_test, y_pred, pos_label=0), 4),
            "Recall (Nhãn 0)": round(recall_score(y_test, y_pred, pos_label=0), 4),
            "F1-score (Nhãn 0)": round(f1_score(y_test, y_pred, pos_label=0), 4),
            "PR-AUC (Nhãn 0)": round(average_precision_score(1 - y_test, 1 - y_prob), 4)
        })
    except Exception as e:
        print(f"❌ Lỗi với strategy {strategy}: {e}")

df_sampling = pd.DataFrame(results)
print(df_sampling.to_string(index=False))



 sampling_strategy Tỷ lệ lớp 1 : lớp 0 sau sampling  Precision (Nhãn 0)  Recall (Nhãn 0)  F1-score (Nhãn 0)  PR-AUC (Nhãn 0)
              0.50                285 : 142 (2 : 1)                 1.0           0.9167             0.9565           0.9936
              0.75                285 : 213 (4 : 3)                 1.0           0.9167             0.9565           0.9936
              1.00                285 : 285 (1 : 1)                 1.0           0.9167             0.9565           0.9936


# **Cau5**

In [46]:
samplers = {
    "Random Over-Sampling": RandomOverSampler(random_state=42),
    "Random Under-Sampling": RandomUnderSampler(random_state=42),
    "SMOTE": SMOTE(random_state=42)
}

results = []

for name, sampler in samplers.items():
    pipeline = ImbPipeline([
        ('scaler', StandardScaler()),
        ('sampler', sampler),
        ('classifier', LogisticRegression(random_state=42, max_iter=1000))
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Kỹ thuật Sampling": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1-score": round(f1_score(y_test, y_pred), 4),
        "PR-AUC": round(average_precision_score(y_test, y_prob), 4)
    })

df_sampling_compare = pd.DataFrame(results)
print(df_sampling_compare.to_string(index=False))

    Kỹ thuật Sampling  Accuracy  Precision  Recall  F1-score  PR-AUC
 Random Over-Sampling    0.9881     0.9863  1.0000    0.9931  0.9998
Random Under-Sampling    0.9762     1.0000  0.9722    0.9859  0.9998
                SMOTE    0.9881     0.9863  1.0000    0.9931  0.9998


# **Cau6**

In [49]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, average_precision_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import pandas as pd

# Định nghĩa các bộ phân loại cần so sánh
classifiers = {
    "KNN (k = 5)": KNeighborsClassifier(n_neighbors=5),
    "SVM (RBF)": SVC(kernel='rbf', probability=True, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

results = []

for clf_name, clf in classifiers.items():
    # Trường hợp 1: CÓ StandardScaler trong Pipeline
    pipe_scaled = ImbPipeline([
        ('scaler', StandardScaler()),
        ('sampler', SMOTE(random_state=42)),
        ('classifier', clf)
    ])
    pipe_scaled.fit(X_train, y_train)
    y_pred_scaled = pipe_scaled.predict(X_test)
    y_prob_scaled = pipe_scaled.predict_proba(X_test)[:, 1] if hasattr(pipe_scaled, "predict_proba") else pipe_scaled.decision_function(X_test)

    f1_s = f1_score(y_test, y_pred_scaled)
    pr_auc_s = average_precision_score(y_test, y_prob_scaled)

    # Trường hợp 2: KHÔNG CÓ StandardScaler trong Pipeline
    pipe_unscaled = ImbPipeline([
        ('sampler', SMOTE(random_state=42)),
        ('classifier', clf)
    ])
    pipe_unscaled.fit(X_train, y_train)
    y_pred_unscaled = pipe_unscaled.predict(X_test)
    y_prob_unscaled = pipe_unscaled.predict_proba(X_test)[:, 1] if hasattr(pipe_unscaled, "predict_proba") else pipe_unscaled.decision_function(X_test)

    f1_u = f1_score(y_test, y_pred_unscaled)
    pr_auc_u = average_precision_score(y_test, y_prob_unscaled)

    # Tính độ chênh lệch F1-score
    diff_f1 = f1_s - f1_u

    results.append({
        "Bộ phân loại": clf_name,
        "F1 (Có Scaler)": round(f1_s, 4),
        "F1 (Không Scaler)": round(f1_u, 4),
        "Chênh lệch F1": round(diff_f1, 4),
        "PR-AUC (Có Scaler)": round(pr_auc_s, 4),
        "PR-AUC (Không Scaler)": round(pr_auc_u, 4)
    })

df_scaler_compare = pd.DataFrame(results)
print(df_scaler_compare.to_string(index=False))

 Bộ phân loại  F1 (Có Scaler)  F1 (Không Scaler)  Chênh lệch F1  PR-AUC (Có Scaler)  PR-AUC (Không Scaler)
  KNN (k = 5)          0.9930             0.9793         0.0137               0.998                 0.9839
    SVM (RBF)          0.9931             0.9583         0.0348               1.000                 0.9953
Decision Tree          0.9722             0.9793        -0.0071               0.969                 0.9710
